# 🎤 Detección de Anomalías Vocales en Tiempo Real

## Extensión del Vocal Pitch Analysis Pipeline

**Integrantes:** Balda Javier · Caracoix Juan · Casas Facundo  
**Institución:** Universidad Católica Argentina  
**Materia:** Análisis y Procesamiento de Datos Streaming  
**Base:** Challenge Streaming — Vocal Pitch Analysis

---

## 📌 Descripción

Este notebook extiende el pipeline de análisis vocal desarrollado en el challenge anterior
incorporando un módulo de **detección de anomalías en tiempo real**.

Mientras el pipeline original detecta *qué nota canta el intérprete*, este módulo detecta
*qué eventos técnicamente relevantes ocurren*: saltos bruscos de registro, clímax vocales,
caídas de intensidad, inestabilidad de pitch y silencios estructurales.

### Anomalías detectadas

| Tipo | Descripción | Aplicación |
|------|-------------|------------|
| `BREAK_VOZ` | Salto brusco de pitch ≥5 semitonos entre frames | Cambio chest→head voice, quiebre |
| `CLIMAX_VOCAL` | Nota aguda sostenida ≥8 frames consecutivos | Momentos de mayor exigencia técnica |
| `CAIDA_INTENS` | Caída de intensidad ≥50% en ventana corta | Cortes, dinámicas abruptas |
| `AGUDO_EXTREMO` | Primera nota en registro sobreagudo (>C6) | Hito técnico de la canción |
| `SILENCIO_LARGO` | >15 frames sin voz consecutivos | Pausas estructurales |
| `INESTABILIDAD` | CV de Hz elevado en ventana corta | Vibrato excesivo, desafinación |

### Arquitectura del sistema

```
Audio (WAV)
    │
    ▼
pipeline.py ──────────────────────────────────────────────────┐
  Demucs (separación vocal)                                    │
  pYIN/torchcrepe (pitch frame a frame, 50ms)                  │
  RMS (intensidad)                                             │
    │                                                          │
    ▼                                                          │
frames: [{time, note, hz, intensity, register, is_voiced}, ...]│
    │                                                          │
    ▼                                                          │
anomaly_detector.py  ←─────────── EXTENSIÓN ──────────────────┘
  VocalAnomalyDetector.detect()
  ├── _detect_break_voz()        ventana 2 frames
  ├── _detect_climax_vocal()     ventana 8–∞ frames
  ├── _detect_caida_intensidad() ventana 10 frames
  ├── _detect_silencio_largo()   streak counter
  ├── _detect_agudo_extremo()    bandera global
  └── _detect_inestabilidad()    ventana 8 frames, CV
    │
    ▼
anomalies.json  +  dashboard interactivo HTML
```

## Sección 1 · Instalación de dependencias

In [ ]:
# Dependencias del pipeline original
!pip install librosa numpy scipy pandas matplotlib -q

# Dependencias adicionales para el detector
# (solo numpy — ya instalado)

# Opcional: para análisis avanzado con Isolation Forest
!pip install scikit-learn -q

print('✅ Dependencias instaladas.')

## Sección 2 · Carga del módulo detector

In [ ]:
# Si subiste anomaly_detector.py a Colab, importarlo directamente.
# Si no, definirlo inline copiando el contenido del archivo.

# Opción A: desde archivo
# from anomaly_detector import VocalAnomalyDetector, summarize, print_report

# Opción B: verificar que está en el path
import importlib.util, sys

spec = importlib.util.spec_from_file_location('anomaly_detector', 'anomaly_detector.py')
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

VocalAnomalyDetector = mod.VocalAnomalyDetector
summarize            = mod.summarize
print_report         = mod.print_report
AnomalyEvent         = mod.AnomalyEvent

print('✅ VocalAnomalyDetector cargado.')

## Sección 3 · Carga de frames del pipeline original

In [ ]:
import pandas as pd
import json

# ── Opción A: desde CSV (results/realtime_frames.csv del challenge) ──
CSV_PATH  = 'results/realtime_frames.csv'
JSON_PATH = 'vocal_analysis/output/frame_analysis.json'

def load_csv(path):
    df = pd.read_csv(path)
    frames = []
    for _, row in df.iterrows():
        frames.append({
            'time':      float(row['time_s']),
            'note':      row['note'] if pd.notna(row['note']) else None,
            'hz':        float(row['hz']) if pd.notna(row['hz']) else None,
            'intensity': float(row['intensity']) if pd.notna(row['intensity']) else 0.0,
            'register':  row['register'],
            'is_voiced': str(row['is_voiced']).lower() == 'true',
        })
    return frames

def load_json(path):
    with open(path) as f:
        return json.load(f)

import os
if os.path.exists(CSV_PATH):
    frames = load_csv(CSV_PATH)
    print(f'✅ Frames cargados desde CSV: {len(frames)} frames')
elif os.path.exists(JSON_PATH):
    frames = load_json(JSON_PATH)
    print(f'✅ Frames cargados desde JSON: {len(frames)} frames')
else:
    print('⚠️  Archivos no encontrados. Generando datos sintéticos de demo...')
    # Generar mini-dataset sintético para demostración
    import numpy as np
    
    sequence = [
        ('A3',0,5,220,0.5,'medio-grave'),
        ('E5',5,12,659,0.8,'agudo'),
        ('E5',12,25,659,0.9,'agudo'),      # clímax vocal
        ('A2',25,30,110,0.4,'grave'),       # nota grave extrema
        (None,30,32,None,0.0,'silencio'),   # silencio
        (None,32,34,None,0.0,'silencio'),
        ('C6',34,40,1047,1.0,'sobreagudo'),  # agudo extremo + clímax
        ('G4',40,45,392,0.6,'medio-agudo'),
    ]
    frames = []
    t = 0.0
    for note, t_s, t_e, hz, base_int, reg in sequence:
        n_frames = int((t_e - t_s) / 0.05)
        for i in range(n_frames):
            jitter = np.random.normal(0, hz * 0.01) if hz else 0
            frames.append({
                'time':      round(t_s + i * 0.05, 3),
                'note':      note,
                'hz':        round(hz + jitter, 2) if hz else None,
                'intensity': round(base_int * 20 + np.random.normal(0, 1), 4),
                'register':  reg,
                'is_voiced': note is not None,
            })
    print(f'✅ Dataset sintético generado: {len(frames)} frames')

# Vista previa
df_preview = pd.DataFrame(frames[:5])
print('\nPrimeros 5 frames:')
display(df_preview)

## Sección 4 · Detección de anomalías

In [ ]:
# Instanciar detector y ejecutar
detector = VocalAnomalyDetector(frames)
events   = detector.detect()

print(f'\n✅ Detección completada: {len(events)} eventos encontrados')

# Reporte en consola
print_report(events)

In [ ]:
# Resumen estadístico
summary = summarize(events)

print('=== RESUMEN ===')
print(f"Total eventos  : {summary['total']}")
print(f"Por tipo       : {summary['por_tipo']}")
print(f"Por severidad  : {summary['por_severidad']}")
if events:
    print(f"Primer evento  : {summary['primer_evento']}s")
    print(f"Último evento  : {summary['ultimo_evento']}s")

In [ ]:
# Convertir eventos a DataFrame para análisis
df_events = pd.DataFrame([e.to_dict() for e in events])
print(f'\nDataFrame de eventos ({len(df_events)} filas x {len(df_events.columns)} cols):')
display(df_events.head(15))

## Sección 5 · Análisis exploratorio de los eventos

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Figura 1: Timeline de eventos sobre el pitch ──────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
fig.suptitle('Detección de anomalías vocales — S.O.S (Bogdan Shuvalov)', fontsize=14, fontweight='bold')

voiced_frames = [f for f in frames if f.get('is_voiced') and f.get('hz')]
times_voiced  = [f['time'] for f in voiced_frames]
hz_voiced     = [f['hz']   for f in voiced_frames]
ints_voiced   = [f['intensity'] for f in voiced_frames]

# Subplot 1: pitch timeline con eventos superpuestos
ax1 = axes[0]
ax1.plot(times_voiced, hz_voiced, color='steelblue', linewidth=0.8, alpha=0.7, label='Pitch (Hz)')
ax1.set_ylabel('Frecuencia (Hz)', fontsize=11)
ax1.set_title('Pitch con eventos detectados', fontsize=11)
ax1.grid(True, alpha=0.3)

# Colores por tipo de evento
COLORS = {
    'BREAK_VOZ':     '#e74c3c',
    'CLIMAX_VOCAL':  '#f39c12',
    'CAIDA_INTENS':  '#3498db',
    'AGUDO_EXTREMO': '#9b59b6',
    'SILENCIO_LARGO':'#2ecc71',
    'INESTABILIDAD': '#e67e22',
}

if not df_events.empty:
    for tipo, color in COLORS.items():
        subset = df_events[df_events['tipo'] == tipo]
        if not subset.empty:
            # Solo mostrar hasta 20 marcadores por tipo para no saturar
            sample = subset.sample(min(20, len(subset)), random_state=42)
            for _, ev in sample.iterrows():
                ax1.axvline(x=ev['time'], color=color, alpha=0.4, linewidth=1.2)
            ax1.scatter(subset['time'], [max(hz_voiced)*1.05]*len(subset),
                       c=color, s=20, marker='v', alpha=0.6)

# Leyenda
patches = [mpatches.Patch(color=c, label=t, alpha=0.7) for t, c in COLORS.items()]
ax1.legend(handles=patches, loc='upper left', fontsize=8, ncol=3)

# Subplot 2: intensidad
ax2 = axes[1]
ax2.fill_between(times_voiced, ints_voiced, alpha=0.4, color='steelblue')
ax2.plot(times_voiced, ints_voiced, color='steelblue', linewidth=0.6)

# Marcar caídas de intensidad
if not df_events.empty:
    caidas = df_events[df_events['tipo'] == 'CAIDA_INTENS']
    if not caidas.empty:
        ax2.scatter(caidas['time'], [0]*len(caidas), c='#e74c3c', s=30, marker='^',
                   zorder=5, label='Caída intens.', alpha=0.8)

ax2.set_xlabel('Tiempo (s)', fontsize=11)
ax2.set_ylabel('Intensidad RMS', fontsize=11)
ax2.set_title('Intensidad vocal con caídas marcadas', fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('anomaly_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figura guardada: anomaly_timeline.png')

In [ ]:
# ── Figura 2: Distribución de eventos por tipo y severidad ────────
if not df_events.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('Estadísticas de anomalías detectadas', fontsize=13, fontweight='bold')

    # Barras por tipo
    tipo_counts = df_events['tipo'].value_counts()
    colors_bar  = [COLORS.get(t, '#999') for t in tipo_counts.index]
    axes[0].barh(tipo_counts.index, tipo_counts.values, color=colors_bar, alpha=0.8)
    axes[0].set_xlabel('Cantidad de eventos')
    axes[0].set_title('Eventos por tipo')
    axes[0].grid(True, alpha=0.3, axis='x')
    for i, v in enumerate(tipo_counts.values):
        axes[0].text(v + 0.5, i, str(v), va='center', fontsize=9)

    # Torta por severidad
    sev_counts = df_events['severidad'].value_counts()
    sev_colors = {'alta': '#e74c3c', 'media': '#f39c12', 'baja': '#2ecc71'}
    axes[1].pie(
        sev_counts.values,
        labels=sev_counts.index,
        colors=[sev_colors.get(s, '#999') for s in sev_counts.index],
        autopct='%1.0f%%',
        startangle=140,
        textprops={'fontsize': 11}
    )
    axes[1].set_title('Distribución por severidad')

    # Histograma temporal (densidad de eventos en el tiempo)
    axes[2].hist(df_events['time'], bins=30, color='steelblue', alpha=0.7, edgecolor='white')
    axes[2].set_xlabel('Tiempo (s)')
    axes[2].set_ylabel('Frecuencia')
    axes[2].set_title('Densidad temporal de eventos')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('anomaly_stats.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Figura guardada: anomaly_stats.png')

## Sección 6 · Análisis avanzado: Isolation Forest

In [ ]:
from sklearn.ensemble import IsolationForest
import pandas as pd
import numpy as np

# ── Preparar features por frame para Isolation Forest ────────────
df_frames = pd.DataFrame([
    f for f in frames if f.get('is_voiced') and f.get('hz')
])

# Features derivadas
df_frames = df_frames.sort_values('time').reset_index(drop=True)
df_frames['hz_diff']        = df_frames['hz'].diff().abs().fillna(0)
df_frames['intensity_diff'] = df_frames['intensity'].diff().abs().fillna(0)
df_frames['hz_rolling_std'] = df_frames['hz'].rolling(8, min_periods=1).std().fillna(0)

features = ['hz', 'intensity', 'hz_diff', 'intensity_diff', 'hz_rolling_std']
X = df_frames[features].values

# Entrenar Isolation Forest
iso = IsolationForest(contamination=0.05, random_state=42)
df_frames['anomaly_iso'] = iso.fit_predict(X)  # -1 = anomalía, 1 = normal
df_frames['anomaly_score'] = iso.decision_function(X)  # score (menor = más anómalo)

n_anomalies = (df_frames['anomaly_iso'] == -1).sum()
print(f'✅ Isolation Forest detectó {n_anomalies} frames anómalos '
      f'({n_anomalies/len(df_frames)*100:.1f}% del total con voz)')

# Top 10 frames más anómalos
top_anom = df_frames[df_frames['anomaly_iso'] == -1].nsmallest(10, 'anomaly_score')
print('\nTop 10 frames más anómalos:')
display(top_anom[['time','note','hz','intensity','hz_diff','anomaly_score']].round(3))

In [ ]:
# ── Comparación: reglas vs Isolation Forest ───────────────────────
rule_times = set(round(e.time, 1) for e in events)
iso_times  = set(round(t, 1) for t in df_frames[df_frames['anomaly_iso'] == -1]['time'])

only_rules = rule_times - iso_times
only_iso   = iso_times - rule_times
both       = rule_times & iso_times

print(f'Detectados solo por reglas:         {len(only_rules)}')
print(f'Detectados solo por Isolation Forest: {len(only_iso)}')
print(f'Detectados por ambos métodos:         {len(both)}')
print(f'\nCobertura combinada:  {len(rule_times | iso_times)} momentos únicos')

## Sección 7 · Exportar resultados y dashboard

In [ ]:
import json
from pathlib import Path

# Guardar JSON de anomalías
summary = summarize(events)

# Agregar resultados de Isolation Forest al summary
summary['isolation_forest'] = {
    'total_anomalos': int(n_anomalies),
    'porcentaje':     round(n_anomalies / len(df_frames) * 100, 1),
    'top_frames': df_frames[df_frames['anomaly_iso'] == -1]
                   .nsmallest(20, 'anomaly_score')[['time','note','hz','anomaly_score']]
                   .to_dict('records')
}

Path('results').mkdir(exist_ok=True)
with open('results/anomalies.json', 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False, default=str)

print('✅ Guardado: results/anomalies.json')

# Exportar CSV de eventos
if not df_events.empty:
    df_events.to_csv('results/anomaly_events.csv', index=False)
    print('✅ Guardado: results/anomaly_events.csv')

print('\nResumen final:')
print(f'  Eventos por reglas:  {summary["total"]}')
print(f'  Frames anómalos IF:  {summary["isolation_forest"]["total_anomalos"]}')
print(f'  Tipos detectados:    {list(summary["por_tipo"].keys())}')

In [ ]:
# ── Descargar archivos generados (en Colab) ────────────────────────
try:
    from google.colab import files
    files.download('results/anomalies.json')
    files.download('anomaly_timeline.png')
    files.download('anomaly_stats.png')
    print('✅ Archivos descargados.')
except Exception:
    print('ℹ️  No estás en Colab — los archivos están en results/')
    import os
    for f in ['results/anomalies.json', 'anomaly_timeline.png', 'anomaly_stats.png']:
        if os.path.exists(f):
            print(f'  ✓ {f}')

## Sección 8 · Dashboard interactivo inline

Genera el dashboard HTML directamente en el notebook (idéntico al enfoque del challenge original).

In [ ]:
# Leer el dashboard HTML generado
import os
DASHBOARD_PATH = 'results/anomaly_dashboard.html'
if os.path.exists(DASHBOARD_PATH):
    from IPython.display import IFrame
    display(IFrame(src=DASHBOARD_PATH, width='100%', height='800'))
else:
    print('Generá primero el dashboard ejecutando: python anomaly_detector.py --csv results/realtime_frames.csv')
    print('Luego abrí results/anomaly_dashboard.html en el navegador.')

---

## 📋 Conclusiones

### ¿Qué detectamos en S.O.S — Bogdan Shuvalov?

| Hallazgo | Tiempo | Descripción |
|----------|--------|-------------|
| Agudo extremo | 59.7s | Primera nota sobreaguda: C6 (1054 Hz) |
| Mayor densidad de clímax | 59–70s | Zona de mayor exigencia vocal |
| Silencio estructural | 44s | Pausa de 0.8s entre secciones |
| Caída máxima | 90s | Caída final del 93% |
| Inestabilidad más alta | 21s | CV=0.183 en transición a B4 |

### Diferencia clave entre métodos

- **Reglas basadas en dominio**: detectan eventos *semánticamente* definidos ("clímax vocal",
  "primera nota sobreaguda"). Son interpretables y ajustables.

- **Isolation Forest**: detecta anomalías *estadísticas* sin conocimiento del dominio.
  Complementa a las reglas capturando patrones inesperados.

### Extensiones posibles

1. Comparar estos eventos entre múltiples canciones/artistas (Idea C)
2. Usar los eventos como features para un clasificador de tipo vocal (Idea A)
3. Integrar con Kafka para detección en tiempo real durante la grabación (Idea B)